In [0]:
from databricks.sdk import WorkspaceClient

DEFAULT_AS_OF = spark.sql(
    "SELECT date_format(from_utc_timestamp(current_timestamp(), 'America/Toronto'), 'yyyy-MM-dd') AS d"
).first()["d"]

dbutils.widgets.text("catalog", "serverless_nv91sa_catalog")
dbutils.widgets.text("team", "team1")
dbutils.widgets.text("as_of_date", DEFAULT_AS_OF)

catalog = dbutils.widgets.get("catalog")
team = dbutils.widgets.get("team")
endpoint_name = f"{team}_sales_agent_gateway"
DIRECT_FMAPI = "databricks-claude-sonnet-4-6"

workspace = WorkspaceClient()
endpoint_exists = False
try:
    workspace.serving_endpoints.get(endpoint_name)
    endpoint_exists = True
except Exception as exc:
    # TODO: if the workspace uses a different API for Unity AI Gateway model services, swap this probe to the supported surface.
    print(f"Gateway probe note: {exc}")

# The deployed app must use the gateway endpoint as a resource, not the direct FMAPI endpoint.
app_endpoints = []
try:
    app = workspace.apps.get(f"{team}-sales-agent")
    app_endpoints = [r.serving_endpoint.name for r in (app.resources or []) if r.serving_endpoint]
except Exception as exc:
    print(f"App probe note: {exc}")
code_ok = bool(app_endpoints) and DIRECT_FMAPI not in app_endpoints
status = "PASS" if endpoint_exists and code_ok else "FAIL"
metric_value = 1.0 if endpoint_exists and code_ok else 0.0

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.ops.milestones (team STRING, round INT, status STRING, metric_name STRING, metric_value DOUBLE, ts TIMESTAMP)")
spark.sql(
    f"INSERT INTO {catalog}.ops.milestones VALUES ('{team}', 4, '{status}', 'gateway_ready', {metric_value}, current_timestamp())"
)

print({"status": status, "endpoint_exists": endpoint_exists, "app_endpoints": app_endpoints, "app_uses_gateway": code_ok})
